# Alpha and beta diversity patterns

Diversity measures for the abundance feature table.

In [ ]:
import pandas as pd
%pip install rpy2
%load_ext rpy2.ipython

In [ ]:
data_dir = 'updog_data'

In [ ]:
!wget -O "$data_dir/updog-feature-table.qza" "https://polybox.ethz.ch/index.php/s/3NwEzFbbXSxZYxX/download"

In [ ]:
!wget -O "$data_dir/updog_metadata.tsv" "https://polybox.ethz.ch/index.php/s/pna5PZy62SfGcq5/download"

## 1. Alpha Rarefaction

In [ ]:
! qiime diversity alpha-rarefaction \
    --i-table $data_dir/updog-feature-table.qza \
    --p-max-depth 10000 \
    --m-metadata-file $data_dir/updog_metadata.tsv \
    --o-visualization $data_dir/alpha-rarefaction.qzv

In [ ]:
Visualization.load(f"{data_dir}/alpha-rarefaction.qzv")

## 2. Core metrics

In [ ]:
! qiime diversity core-metrics \
  --i-phylogeny rooted-tree.qza \
  --i-table table.qza \
  --p-sampling-depth <your-depth> \
  --m-metadata-file metadata.tsv \
  --output-dir core-metrics-results


## 3. Significance
### 3.1. Shannon's index

In [ ]:
! qiime diversity alpha-group-significance \
  --i-alpha-diversity $data_dir/core-metrics-results/shannon_vector.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --o-visualization $data_dir/core-metrics-results/shannon-group-significance.qzv

### 3.2. Bray-Curtis PERMANOVA for Lifestyle

In [ ]:
! qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results/bray_curtis_distance_matrix.qza \
  --m-metadata-file updog_metadata.tsv \
  --m-metadata-column Lifestyle \
  --o-visualization core-metrics-results/bray-curtis-lifestyle-significance.qzv \
  --p-pairwise

### 3.3. Bray-Curtis PERMANOVA for other covariables

In [ ]:
# By Country
qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results/bray_curtis_distance_matrix.qza \
  --m-metadata-file updog_metadata.tsv \
  --m-metadata-column Country \
  --o-visualization core-metrics-results/bray-curtis-country-significance.qzv \
  --p-pairwise

# By Sex
qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results/bray_curtis_distance_matrix.qza \
  --m-metadata-file updog_metadata.tsv \
  --m-metadata-column Sex \
  --o-visualization core-metrics-results/bray-curtis-sex-significance.qzv \
  --p-pairwise

# By Age group (if Age is categorical; if continuous, see below)
qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results/bray_curtis_distance_matrix.qza \
  --m-metadata-file updog_metadata.tsv \
  --m-metadata-column AgeGroup \
  --o-visualization core-metrics-results/bray-curtis-agegroup-significance.qzv \
  --p-pairwise

# By BMI category
qiime diversity beta-group-significance \
  --i-distance-matrix core-metrics-results/bray_curtis_distance_matrix.qza \
  --m-metadata-file updog_metadata.tsv \
  --m-metadata-column BMIgroup \
  --o-visualization core-metrics-results/bray-curtis-bmigroup-significance.qzv \
  --p-pairwise

### 3.4. Multivariate PERMANOVA

In [ ]:
! qiime tools export \
  --input-path core-metrics-results/bray_curtis_distance_matrix.qza \
  --output-path exported-bray

In [ ]:
# Load distance matrix - dunno if needed this cell
dist = pd.read_csv("exported-bray/distance-matrix.tsv", sep='\t', index_col=0)
dist = dist.drop(index='#SampleID', errors='ignore')  # clean header if needed
dm = DistanceMatrix(dist.values, ids=dist.index)

# Load metadata
meta = pd.read_csv("updog_metadata.tsv", sep='\t', index_col=0)
meta = meta.loc[dm.ids]  # ensure same sample order

In [ ]:
# to test lifestyle
res = permanova(dm, meta, column='Lifestyle', permutations=999)
print(res)

In [ ]:
%load_ext rpy2.ipython

%%R
library(vegan)
dist <- as.dist(read.table("exported-bray/distance-matrix.tsv", header=TRUE, row.names=1))
meta <- read.table("updog_metadata.tsv", header=TRUE, sep="\t", row.names=1)
adonis2(dist ~ Lifestyle + Country + Sex + Age + BMI, data=meta)